<a id='8'></a>
## 8 — Live API Connection Test

This tests that your chosen backend works correctly.  
Set `LLM_BACKEND` in your `.env` to `ollama`, `gemini`, or `together`,  
then run the cell below to confirm your environment is fully connected.


<a id='1'></a>
## 1 — Environment Setup

Before anything else, verify your environment is correct.  
Your `.env` file should be at `D:\AI_Engineer-RAG\.env`.  
Set `LLM_BACKEND` to your preferred backend:

```
# .env — pick ONE backend
LLM_BACKEND=ollama          # local Ollama (no API key needed)
# LLM_BACKEND=gemini        # Google Gemini (needs GEMINI_API_KEY)
# LLM_BACKEND=together      # Together.ai  (needs TOGETHER_API_KEY)

OLLAMA_MODEL=qwen2.5:7b
GEMINI_API_KEY=your_gemini_key_here
TOGETHER_API_KEY=your_together_key_here
```


In [64]:
# ── Check Python version ──────────────────────────────────────────────────────
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 11), "Need Python 3.11+"

Python version: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]


In [65]:
# ── Load environment variables from .env ──────────────────────────────────────
# python-dotenv reads the .env file and puts the values into os.environ
from dotenv import load_dotenv
import os

load_dotenv()  # looks for .env in the current folder or parent folders

# Show which backend is active
backend = os.environ.get('LLM_BACKEND', 'ollama')
print(f'✅ LLM_BACKEND: {backend}')

# Check relevant key depending on backend
if backend == 'together':
    key = os.environ.get('TOGETHER_API_KEY', '')
    print(f'✅ TOGETHER_API_KEY loaded: {key[:8]}...' if key else '❌ TOGETHER_API_KEY not found — check your .env file')
elif backend == 'gemini':
    key = os.environ.get('GEMINI_API_KEY', '')
    print(f'✅ GEMINI_API_KEY loaded: {key[:8]}...' if key else '❌ GEMINI_API_KEY not found — check your .env file')
elif backend == 'ollama':
    host = os.environ.get('OLLAMA_HOST', 'http://localhost:11434')
    print(f'✅ Ollama backend — host: {host} (make sure `ollama serve` is running)')


✅ LLM_BACKEND: ollama
✅ Ollama backend — host: http://localhost:11434 (make sure `ollama serve` is running)


---
<a id='2'></a>
## 2 — Lists

<a id='2-1'></a>
### 2.1 Basics

Lists are the most common data structure in this course.  
Retrieved documents, conversation histories, and chunks are all stored as lists.

In [66]:
# ── Basic list operations ─────────────────────────────────────────────────────
# In this course, lists of strings represent retrieved documents
documents = ["RAG stands for Retrieval Augmented Generation.",
             "LLMs have a training cutoff date.",
             "Vector databases store embeddings."]

print(f"Documents: {documents}")
print(f"First document: {documents[0]}")
print(f"Last document:  {documents[-1]}")
print(f"Total documents: {len(documents)}")

Documents: ['RAG stands for Retrieval Augmented Generation.', 'LLMs have a training cutoff date.', 'Vector databases store embeddings.']
First document: RAG stands for Retrieval Augmented Generation.
Last document:  Vector databases store embeddings.
Total documents: 3


In [67]:
# ── append vs extend ─────────────────────────────────────────────────────────
# append adds ONE item; extend adds ALL items from another list

more_docs = ["Chunking splits documents into smaller pieces.",
             "Embeddings map text to vectors."]

documents.extend(more_docs)       # adds both items
print(f"After extend: {len(documents)} documents")

documents.append("One more doc.") # adds exactly one item
print(f"After append: {len(documents)} documents")

After extend: 5 documents
After append: 6 documents


In [68]:
# ── Slicing ───────────────────────────────────────────────────────────────────
# Retrievers often return the top-k results — slicing is how you take top-k
top_3 = documents[:3]
print(f"Top 3 retrieved docs: {top_3}")

Top 3 retrieved docs: ['RAG stands for Retrieval Augmented Generation.', 'LLMs have a training cutoff date.', 'Vector databases store embeddings.']


<a id='2-2'></a>
### 2.2 List Comprehensions

List comprehensions replace for-loops for building lists.  
You'll use them constantly to transform and filter documents.

In [69]:
# ── Basic list comprehension ──────────────────────────────────────────────────
# Add a label number to each document (like a retriever ranking)
labeled_docs = [f"[Doc {i+1}] {doc}" for i, doc in enumerate(documents[:3])]

for doc in labeled_docs:
    print(doc)

[Doc 1] RAG stands for Retrieval Augmented Generation.
[Doc 2] LLMs have a training cutoff date.
[Doc 3] Vector databases store embeddings.


In [71]:
# ── Conditional list comprehension ───────────────────────────────────────────
# Keep only documents that mention 'RAG' — like a simple keyword filter
rag_docs = [doc for doc in documents if "RAG" in doc]

print(f"Documents about RAG:")
for doc in rag_docs:
    print(f"  - {doc}")

Documents about RAG:
  - RAG stands for Retrieval Augmented Generation.


In [72]:
# ── Equivalent for-loop version (for comparison) ─────────────────────────────
rag_docs_loop = []
for doc in documents:
    if "RAG" in doc:
        rag_docs_loop.append(doc)

# Both produce the same result
print(rag_docs == rag_docs_loop)

True


---
<a id='3'></a>
## 3 — Dictionaries

<a id='3-1'></a>
### 3.1 Basics

Dictionaries are the main data format for LLM API messages.  
Every message you send to an LLM is a `{role, content}` dictionary.

In [73]:
# ── Basic dictionary ─────────────────────────────────────────────────────────
# A single LLM message — this is the exact format used by Together.ai and OpenAI
message = {
    "role": "user",
    "content": "What is RAG?"
}

print(f"Role:    {message['role']}")
print(f"Content: {message['content']}")

Role:    user
Content: What is RAG?


In [74]:
# ── .get() — safe access with a default ──────────────────────────────────────
# Use .get() when a key might not exist, to avoid KeyError
role = message.get("role", "unknown")
source = message.get("source", "not provided")  # key doesn't exist — returns default

print(f"role:   {role}")
print(f"source: {source}")

role:   user
source: not provided


<a id='3-2'></a>
### 3.2 The `.items()` Method

In [75]:
# ── Iterating with .items() ───────────────────────────────────────────────────
# Useful for inspecting API responses
api_response = {
    "role": "assistant",
    "content": "RAG stands for Retrieval Augmented Generation.",
    "model": "Qwen/Qwen3.5-9B",
    "tokens_used": 42
}

for key, value in api_response.items():
    print(f"  {key}: {value}")

  role: assistant
  content: RAG stands for Retrieval Augmented Generation.
  model: Qwen/Qwen3.5-9B
  tokens_used: 42


In [76]:
# ── Iterating directly iterates over keys only ────────────────────────────────
for key in api_response:
    print(key)

role
content
model
tokens_used


---
<a id='4'></a>
## 4 — f-strings and String Templates

<a id='4-1'></a>
### 4.1 f-string Basics

f-strings are how you build prompts dynamically.  
Every RAG prompt is built with f-strings.

In [77]:
# ── Basic f-string ────────────────────────────────────────────────────────────
user_name = "AbdelRahman"
topic = "RAG"

greeting = f"Hello, {user_name}! Today we are studying {topic}."
print(greeting)

Hello, AbdelRahman! Today we are studying RAG.


In [78]:
# ── Multi-line f-string with triple quotes ────────────────────────────────────
# This is the exact shape of a RAG prompt
question = "What is retrieval augmented generation?"
context  = "RAG is a technique that improves LLM responses by retrieving relevant documents."

prompt = f"""Answer the question using only the context below.

Context:
{context}

Question:
{question}

Answer:"""

print(prompt)

Answer the question using only the context below.

Context:
RAG is a technique that improves LLM responses by retrieving relevant documents.

Question:
What is retrieval augmented generation?

Answer:


<a id='4-2'></a>
### 4.2 Building Strings from Lists of Dicts

This pattern from Lab 1 is used throughout the course.  
Here it's applied to RAG-specific data.

In [79]:
# ── List of dicts — like retrieved documents with metadata ────────────────────
retrieved_docs = [
    {"id": 1, "score": 0.95, "text": "RAG retrieves relevant documents before generating."},
    {"id": 2, "score": 0.88, "text": "The retriever searches a vector database."},
    {"id": 3, "score": 0.81, "text": "The LLM reads the retrieved docs and generates an answer."},
]

In [80]:
# ── Build the context block for a RAG prompt ──────────────────────────────────
# Using f-string inside list comprehension, then joining with newline
context_lines = [
    f"[Document {doc['id']} | score: {doc['score']}]\n{doc['text']}"
    for doc in retrieved_docs
]

context_block = "\n\n".join(context_lines)
print(context_block)

[Document 1 | score: 0.95]
RAG retrieves relevant documents before generating.

[Document 2 | score: 0.88]
The retriever searches a vector database.

[Document 3 | score: 0.81]
The LLM reads the retrieved docs and generates an answer.


In [81]:
# ── Drop it into a full prompt ────────────────────────────────────────────────
user_question = "How does a RAG system work?"

full_prompt = f"""Answer the question using the retrieved documents below.

Retrieved Documents:
{context_block}

Question: {user_question}

Answer:"""

print(full_prompt)

Answer the question using the retrieved documents below.

Retrieved Documents:
[Document 1 | score: 0.95]
RAG retrieves relevant documents before generating.

[Document 2 | score: 0.88]
The retriever searches a vector database.

[Document 3 | score: 0.81]
The LLM reads the retrieved docs and generates an answer.

Question: How does a RAG system work?

Answer:


---
<a id='5'></a>
## 5 — Functions and Type Hints

The course utilities use type hints like `str`, `List[Dict]`, `float`.  
This section makes sure you can read and write them.

In [82]:
# ── Type hints ────────────────────────────────────────────────────────────────
# Type hints don't enforce types at runtime — they're just documentation
from typing import List, Dict, Optional

def format_documents(docs: List[Dict], max_docs: int = 3) -> str:
    """
    Takes a list of document dicts and returns a single formatted string.
    This is what a real RAG pipeline does before injecting into the prompt.
    
    Args:
        docs:     List of dicts, each with 'id' and 'text' keys
        max_docs: Maximum number of documents to include
    
    Returns:
        A formatted string ready to insert into a prompt
    """
    top_docs = docs[:max_docs]  # take only the top-k
    lines = [f"[Doc {doc['id']}] {doc['text']}" for doc in top_docs]
    return "\n".join(lines)


result = format_documents(retrieved_docs, max_docs=2)
print(result)

[Doc 1] RAG retrieves relevant documents before generating.
[Doc 2] The retriever searches a vector database.


In [83]:
# ── Optional parameters ───────────────────────────────────────────────────────
# Optional[str] means the value can be a string OR None
def build_prompt(question: str,
                 context: str,
                 system_note: Optional[str] = None) -> str:
    """
    Builds a RAG prompt. Optionally prepends a system note.
    """
    base = f"""Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
    
    if system_note:
        return f"Note: {system_note}\n\n{base}"
    return base


# Without system note
print(build_prompt("What is RAG?", "RAG is a retrieval technique."))
print("---")
# With system note
print(build_prompt("What is RAG?", "RAG is a retrieval technique.", system_note="Be concise."))

Context:
RAG is a retrieval technique.

Question: What is RAG?

Answer:
---
Note: Be concise.

Context:
RAG is a retrieval technique.

Question: What is RAG?

Answer:


---
<a id='6'></a>
## 6 — The Message Format — How LLM APIs Work

All LLM APIs (Together.ai, OpenAI, Anthropic) use the same message format.  
Understanding this is the single most important thing in this section.

<a id='6-1'></a>
### 6.1 The role/content Dict

Every message is a dict with two keys:
- `role` — who is speaking: `"system"`, `"user"`, or `"assistant"`
- `content` — what they said

In [84]:
# ── The three roles explained ─────────────────────────────────────────────────

system_message = {
    "role": "system",
    # Sets the LLM's behavior for the whole conversation
    # Only sent once, at the start
    "content": "You are a helpful RAG assistant. Answer only from the provided documents."
}

user_message = {
    "role": "user",
    # What the human is asking
    "content": "What is RAG?"
}

assistant_message = {
    "role": "assistant",
    # The LLM's response — used in multi-turn conversations
    "content": "RAG stands for Retrieval Augmented Generation."
}

# You always send a LIST of these dicts
messages = [system_message, user_message]
print(messages)

[{'role': 'system', 'content': 'You are a helpful RAG assistant. Answer only from the provided documents.'}, {'role': 'user', 'content': 'What is RAG?'}]


<a id='6-2'></a>
### 6.2 Building a Conversation History

For multi-turn chat, you append each exchange to the messages list.  
The LLM sees the full history every time.

In [85]:
# ── Building conversation history ─────────────────────────────────────────────
# Start with system message
conversation = [
    {"role": "system", "content": "You are a RAG expert. Be concise."}
]

def add_user_message(history: List[Dict], text: str) -> List[Dict]:
    """Appends a user message to the conversation history."""
    history.append({"role": "user", "content": text})
    return history

def add_assistant_message(history: List[Dict], text: str) -> List[Dict]:
    """Appends an assistant response to the conversation history."""
    history.append({"role": "assistant", "content": text})
    return history

# Simulate a 2-turn conversation
conversation = add_user_message(conversation, "What is RAG?")
conversation = add_assistant_message(conversation, "RAG is Retrieval Augmented Generation.")
conversation = add_user_message(conversation, "What problem does it solve?")

# Print clearly
for i, msg in enumerate(conversation):
    print(f"[{i}] {msg['role'].upper()}: {msg['content']}")

[0] SYSTEM: You are a RAG expert. Be concise.
[1] USER: What is RAG?
[2] ASSISTANT: RAG is Retrieval Augmented Generation.
[3] USER: What problem does it solve?


---
<a id='7'></a>
## 7 — Prompt Augmentation Pattern (The Core of RAG)

This is the pattern you will write dozens of times in this course.  
A user question + retrieved documents = augmented prompt.

In [86]:
# ── Simulated knowledge base ──────────────────────────────────────────────────
# In the real course, these come from a vector database
knowledge_base = [
    "RAG stands for Retrieval Augmented Generation.",
    "RAG combines a retriever with an LLM to generate grounded answers.",
    "The retriever finds relevant documents from a knowledge base.",
    "The LLM reads the retrieved documents and generates a response.",
    "RAG reduces hallucinations by grounding the LLM in real documents.",
    "Vector databases store embeddings and support semantic search.",
    "LLMs have a training cutoff and cannot access recent information.",
]

In [87]:
# ── Simple keyword retriever (placeholder for vector search) ──────────────────
def simple_retriever(query: str, corpus: List[str], top_k: int = 3) -> List[str]:
    """
    Retrieves the top_k documents that contain any word from the query.
    This is a naive keyword retriever — later modules replace this with
    semantic/vector search.
    """
    query_words = query.lower().split()
    
    # Score each doc by how many query words it contains
    scored = []
    for doc in corpus:
        score = sum(1 for word in query_words if word in doc.lower())
        if score > 0:
            scored.append((score, doc))
    
    # Sort by score descending, return top_k texts
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:top_k]]


# Test it
results = simple_retriever("How does RAG reduce hallucinations?", knowledge_base)
print("Retrieved documents:")
for i, doc in enumerate(results, 1):
    print(f"  [{i}] {doc}")

Retrieved documents:
  [1] RAG reduces hallucinations by grounding the LLM in real documents.
  [2] RAG stands for Retrieval Augmented Generation.
  [3] RAG combines a retriever with an LLM to generate grounded answers.


In [88]:
# ── Augment the prompt ────────────────────────────────────────────────────────
def build_rag_prompt(question: str, retrieved_docs: List[str]) -> str:
    """
    Combines retrieved documents with the user question.
    This is the Augmentation step in Retrieve → Augment → Generate.
    """
    # Number each document
    context = "\n".join([f"[{i+1}] {doc}" for i, doc in enumerate(retrieved_docs)])
    
    prompt = f"""Answer the question using ONLY the documents provided below.
If the answer is not in the documents, say "I don't have that information."

Documents:
{context}

Question: {question}

Answer:"""
    return prompt


question = "How does RAG reduce hallucinations?"
docs = simple_retriever(question, knowledge_base)
prompt = build_rag_prompt(question, docs)

print(prompt)

Answer the question using ONLY the documents provided below.
If the answer is not in the documents, say "I don't have that information."

Documents:
[1] RAG reduces hallucinations by grounding the LLM in real documents.
[2] RAG stands for Retrieval Augmented Generation.
[3] RAG combines a retriever with an LLM to generate grounded answers.

Question: How does RAG reduce hallucinations?

Answer:


In [89]:
# ── Wrap as a messages list (ready to send to the API) ────────────────────────
def rag_messages(question: str, retrieved_docs: List[str]) -> List[Dict]:
    """
    Returns the messages list ready to pass to any LLM API.
    """
    augmented_prompt = build_rag_prompt(question, retrieved_docs)
    return [
        {"role": "system", "content": "You are a helpful assistant that answers from provided documents."},
        {"role": "user",   "content": augmented_prompt}
    ]


messages = rag_messages(question, docs)
for msg in messages:
    print(f"[{msg['role'].upper()}]")
    print(msg['content'])
    print()

[SYSTEM]
You are a helpful assistant that answers from provided documents.

[USER]
Answer the question using ONLY the documents provided below.
If the answer is not in the documents, say "I don't have that information."

Documents:
[1] RAG reduces hallucinations by grounding the LLM in real documents.
[2] RAG stands for Retrieval Augmented Generation.
[3] RAG combines a retriever with an LLM to generate grounded answers.

Question: How does RAG reduce hallucinations?

Answer:



<a id='8'></a>
## 8 — Live API Connection Test

This tests that your chosen backend works correctly.  
Set `LLM_BACKEND` in your `.env` to `ollama`, `gemini`, or `together`,  
then run the cell below to confirm your environment is fully connected.


In [90]:
# ── Direct API connection test ────────────────────────────────────────────────
# Tests whichever backend is set in LLM_BACKEND
from dotenv import load_dotenv
import os

load_dotenv()
backend = os.environ.get('LLM_BACKEND', 'ollama')
print(f'Testing backend: {backend}')

if backend == 'ollama':
    import requests
    host = os.environ.get('OLLAMA_HOST', 'http://localhost:11434')
    model = os.environ.get('OLLAMA_MODEL', 'qwen2.5:7b')
    resp = requests.post(f'{host}/api/chat', json={
        'model': model,
        'messages': [{'role': 'user', 'content': 'Say: API connection successful. Nothing else.'}],
        'stream': False,
        'options': {'num_predict': 20, 'temperature': 0.0}
    }, timeout=60)
    resp.raise_for_status()
    print(resp.json()['message']['content'])

elif backend == 'gemini':
    import google.generativeai as genai
    genai.configure(api_key=os.environ['GEMINI_API_KEY'])
    model_name = os.environ.get('GEMINI_MODEL', 'gemini-2.0-flash')
    model = genai.GenerativeModel(model_name)
    resp = model.generate_content('Say: API connection successful. Nothing else.')
    print(resp.text)

elif backend == 'together':
    from together import Together
    client = Together(api_key=os.environ['TOGETHER_API_KEY'])
    model = os.environ.get('TOGETHER_MODEL', 'Qwen/Qwen2.5-7B-Instruct-Turbo')
    response = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': 'Say: API connection successful. Nothing else.'}],
        max_tokens=20,
        temperature=0.0
    )
    print(response.choices[0].message.content)


Testing backend: ollama



In [91]:
# ── Same call using utils.py helper ──────────────────────────────────────────
# This is how the course notebooks call the API
import sys
sys.path.append("../../")  # so Python can find utils.py in the parent folders

# Uncomment the line below once utils.py is in the right place:
# from utils import generate_with_single_input
# response = generate_with_single_input("What is RAG in one sentence?")
# print(response['content'])

---
<a id='9'></a>
## 9 — Putting It All Together — Mini RAG Skeleton

This is the complete RAG loop in ~20 lines.  
Every lab and project in this course is a more sophisticated version of this.

In [92]:
# ── Complete mini RAG pipeline ────────────────────────────────────────────────
from dotenv import load_dotenv
import os

load_dotenv()
backend = os.environ.get('LLM_BACKEND', 'ollama')


def _llm_call(messages: list, model: str = None, max_tokens: int = 200) -> str:
    """Internal helper — routes to the active backend."""
    if backend == 'ollama':
        import requests
        host = os.environ.get('OLLAMA_HOST', 'http://localhost:11434')
        _model = model or os.environ.get('OLLAMA_MODEL', 'qwen2.5:7b')
        resp = requests.post(f'{host}/api/chat', json={
            'model': _model, 'messages': messages, 'stream': False,
            'options': {'num_predict': max_tokens, 'temperature': 0.0}
        }, timeout=120)
        resp.raise_for_status()
        return resp.json()['message']['content']
    elif backend == 'gemini':
        import google.generativeai as genai
        genai.configure(api_key=os.environ['GEMINI_API_KEY'])
        _model = model or os.environ.get('GEMINI_MODEL', 'gemini-2.0-flash')
        cfg = genai.types.GenerationConfig(max_output_tokens=max_tokens, temperature=0.0)
        sys_msg = next((m['content'] for m in messages if m['role'] == 'system'), None)
        history = [{'role': 'model' if m['role'] == 'assistant' else m['role'], 'parts': [m['content']]}
                   for m in messages if m['role'] != 'system']
        last = history.pop()['parts'][0]
        chat = genai.GenerativeModel(_model, system_instruction=sys_msg, generation_config=cfg).start_chat(history=history)
        return chat.send_message(last).text
    elif backend == 'together':
        from together import Together
        _model = model or os.environ.get('TOGETHER_MODEL', 'Qwen/Qwen2.5-7B-Instruct-Turbo')
        client = Together(api_key=os.environ['TOGETHER_API_KEY'])
        resp = client.chat.completions.create(model=_model, messages=messages,
                                               max_tokens=max_tokens, temperature=0.0,
                                               reasoning={'enabled': False})
        return resp.choices[0].message.content


def rag_pipeline(user_question: str,
                 corpus: List[str],
                 top_k: int = 3,
                 model: str = None) -> str:
    """
    Full RAG pipeline: Retrieve → Augment → Generate.
    Works with ollama, gemini, or together backend (set LLM_BACKEND in .env).

    Args:
        user_question: The question to answer
        corpus:        The knowledge base (list of strings)
        top_k:         How many documents to retrieve
        model:         Override model name (None = use backend default)

    Returns:
        The LLM's answer as a string
    """
    # ── Step 1: RETRIEVE ──────────────────────────────────────────────────────
    retrieved = simple_retriever(user_question, corpus, top_k=top_k)

    # ── Step 2: AUGMENT ───────────────────────────────────────────────────────
    messages = rag_messages(user_question, retrieved)

    # ── Step 3: GENERATE ──────────────────────────────────────────────────────
    return _llm_call(messages, model=model, max_tokens=200)


# ── Run it ────────────────────────────────────────────────────────────────────
question = 'Why does RAG reduce hallucinations?'
answer = rag_pipeline(question, knowledge_base)

print(f'Question: {question}')
print(f'\nAnswer: {answer}')


Question: Why does RAG reduce hallucinations?

Answer: 


In [93]:
# ── Try your own question ─────────────────────────────────────────────────────
my_question = "What is the role of the retriever in RAG?"  # ← change this

my_answer = rag_pipeline(my_question, knowledge_base)
print(f"Q: {my_question}")
print(f"A: {my_answer}")

Q: What is the role of the retriever in RAG?
A: 


---

## ✅ Summary — What You Now Know

| Concept | Used for |
|---|---|
| `load_dotenv()` | Loading your API key safely |
| Lists + slicing | Storing and selecting retrieved documents |
| List comprehensions | Transforming document lists |
| Dicts + `.items()` | Message format, API responses |
| f-strings + triple quotes | Building prompts dynamically |
| Type hints | Reading and writing course utility functions |
| `{role, content}` messages | The universal LLM API format |
| Conversation history | Multi-turn chat |
| Retrieve → Augment → Generate | The full RAG loop |

You are ready for the course labs.

---

*Next: Module 1 Lab 2 — calling the LLM API with `utils.py`*